# Polygon conservative regridder — unstructured mesh + save/load

When cells are arbitrary polygons (ICON triangles, MPAS hexagons, country shapes for
regional aggregation) use `ConservativeRegridder.from_polygons`. The shapely STRtree
handles any polygon layout.

This notebook builds a synthetic Voronoi hex-like mesh with scipy, regrids a structured
source onto it, then persists the regridder so subsequent runs skip the weight build.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from scipy.spatial import Voronoi
import shapely

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder, polygons_from_coords

## Build a hex-like Voronoi mesh

Jittered grid points → Voronoi → clip to the region of interest. Real workflows
would load a pre-built mesh (UGRID, ICON, etc.); the point here is that
`from_polygons` needs only a 1D array of shapely polygons.

In [ ]:
def voronoi_mesh(n_points, bbox, seed=0):
    rng = np.random.default_rng(seed)
    x0, y0, x1, y1 = bbox
    side = int(np.sqrt(n_points))
    xs = np.linspace(x0, x1, side); ys = np.linspace(y0, y1, side)
    pts = np.column_stack([np.repeat(xs, side), np.tile(ys, side)])
    pts += rng.normal(scale=(x1 - x0) / side * 0.25, size=pts.shape)
    halo = np.array([
        [2*x0 - x1, 2*y0 - y1], [2*x1 - x0, 2*y0 - y1],
        [2*x0 - x1, 2*y1 - y0], [2*x1 - x0, 2*y1 - y0],
    ])
    vor = Voronoi(np.concatenate([pts, halo]))
    clip = shapely.box(x0, y0, x1, y1)
    polys, centers = [], []
    for i in range(len(pts)):
        r = vor.regions[vor.point_region[i]]
        if not r or -1 in r:
            continue
        p = shapely.intersection(shapely.Polygon(vor.vertices[r]), clip)
        if p.is_empty or p.geom_type != "Polygon":
            continue
        polys.append(p); centers.append(pts[i])
    return np.array(polys, dtype=object), np.array(centers)

bbox = (-120, -50, 120, 50)
mesh_polys, mesh_centers = voronoi_mesh(n_points=400, bbox=bbox)
print(f"{len(mesh_polys)} cells in the mesh")

## Source — structured lat/lon field

In [ ]:
lat_s = np.linspace(-50, 50, 100, endpoint=False) + 0.5
lon_s = np.linspace(-120, 120, 240, endpoint=False) + 0.5
Lo, La = np.meshgrid(lon_s, lat_s)
field = np.sin(np.deg2rad(Lo) * 2) * np.cos(np.deg2rad(La) * 3)
src = xr.DataArray(
    field,
    dims=("latitude", "longitude"),
    coords={"latitude": lat_s, "longitude": lon_s},
    name="field",
)
src.plot(figsize=(8, 3.3), cmap="RdBu_r", center=0)
plt.title("structured source")
plt.tight_layout()

## Regrid structured → mesh with `from_polygons`

We convert the structured grid into a flat polygon list via `polygons_from_coords`
(row-major, y-slow / x-fast) and flatten the data the same way. Output is a 1D
array indexed by mesh cell.

In [ ]:
grid_polys = polygons_from_coords(lon_s, lat_s)

rgr = ConservativeRegridder.from_polygons(
    grid_polys, mesh_polys,
    source_dim="src_cell",
    target_dim="cell",
)
print(rgr)

src_flat = xr.DataArray(src.values.ravel(), dims=("src_cell",))
mesh_vals = rgr.regrid(src_flat)
mesh_vals

## Plot the regridded field on the mesh

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
patches = [np.asarray(p.exterior.coords) for p in mesh_polys]
pc = PolyCollection(patches, array=mesh_vals.values, cmap="RdBu_r",
                    edgecolor="0.4", lw=0.3, clim=(-1, 1))
ax.add_collection(pc)
ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
ax.set_aspect("equal")
fig.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title(f"regridded onto {len(mesh_polys)}-cell Voronoi mesh")

## Persist the regridder to netCDF

For a fixed source / target pair the weight matrix is the same forever. Persisting
it to disk lets long-running pipelines skip the (expensive) build on restart.

In [ ]:
path = Path(tempfile.gettempdir()) / "mesh_regridder.nc"
rgr.to_netcdf(path)
print(f"wrote {path} ({path.stat().st_size / 1024:.1f} KB)")

# Inspect on-disk metadata (useful for provenance):
with xr.open_dataset(path) as weights:
    for k, v in weights.attrs.items():
        print(f"  {k}: {v}")

## Reload and apply to new data

In [ ]:
rgr2 = ConservativeRegridder.from_netcdf(path)

# Apply to a new structured field of the same shape:
new_src = xr.DataArray(
    np.cos(np.deg2rad(Lo)) * np.sin(np.deg2rad(La) * 2),
    dims=("latitude", "longitude"),
    coords=src.coords,
)
out = rgr2.regrid(xr.DataArray(new_src.values.ravel(), dims=("src_cell",)))

# Sanity check: reloaded regridder produces identical output to the original.
reference = rgr.regrid(xr.DataArray(new_src.values.ravel(), dims=("src_cell",)))
print(f"max diff reloaded vs original: {float(np.abs(out.values - reference.values).max()):.2e}")